# 3D controls, session A (~9.7 h on a T4)

True 3D Born, BBBC024, 60 receivers. Accelerator GPU T4, Internet on, then Save & Run All. Sessions A and B are independent.

In [ ]:
SESSION = "a"          # run "a" and "b" as two separate Kaggle sessions (any order, no value to copy between them)

import os, subprocess, sys, glob, shutil, zipfile
CONFIG = f"thesis1_3d_{SESSION}"
CODE, WORK = "/kaggle/working/thesis-gs", "/kaggle/working"
OUT, VOL = f"{WORK}/{CONFIG}", "/tmp/cytopacq"

shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/muhammadhussainqureshi/thesis-gs", CODE], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lmdb", "tifffile"], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

# BBBC024 ground-truth volumes (synthetic HL60 nuclei)
os.makedirs(VOL, exist_ok=True)
name = "BBBC024_v1_c00_highSNR_foreground.zip"
subprocess.run(["wget", "-q", "-O", f"{VOL}/{name}", f"https://data.broadinstitute.org/bbbc/BBBC024/{name}"], check=True)
zipfile.ZipFile(f"{VOL}/{name}").extractall(VOL)
print(len(glob.glob(f"{VOL}/**/*.ics", recursive=True)), "BBBC024 volumes")

subprocess.run([sys.executable, f"{CODE}/bin/run_experiments.py", f"{CODE}/configs/{CONFIG}.json", "--out", OUT,
                "--data_dir", f"{WORK}/data", "--volumes", VOL, "--budget_hours", "11.3",
                "--zip", f"{WORK}/{CONFIG}_results.zip"])

subprocess.run([sys.executable, f"{CODE}/bin/scatter3d_report.py", OUT, "--out", f"{OUT}/report",
                "--ref2d", f"{CODE}/results/2026-11-10_bounded/scatter2d_10_bounded_gridinit/scatter2d/results.csv"])
for p in glob.glob(f"{OUT}/**/*.pt", recursive=True):
    os.remove(p)
shutil.make_archive(f"{WORK}/{CONFIG}_results", "zip", OUT)
print(f"Download {CONFIG}_results.zip from the Output tab")
